# Junior AI Investigator: fraud referral triage

The fraud engine flags home-care claims; most turn out to be false alarms. For each of the 50
referrals, the AI scores the risk, explains it in plain English and suggests what to do next.
Every fact it cites is checked against the data, and hard evidence (double billing, overlapping
visits, a shared claim number) can never be closed. The investigator reviews the queue in a web
page served by the last cell, and makes the final call.

**In Colab:** add your Groq key under Secrets (🔑, left sidebar) as `GROQ_API_KEY`, then
*Runtime → Run all*. When asked, upload `sample_cases.csv` and `ai_assessments.json` (from `data/`)
and `index.html`, `app.js` and `styles.css` (from `web/`), then click the link the last cell shows.
**Locally:** `pip install -r requirements.txt`, put `GROQ_API_KEY=...` in `.env`, then
`jupyter notebook fraud_triage.ipynb` from the project folder, run all, and open http://localhost:8000.

In [ ]:
import json, os, re, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from html import escape
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field

DATA, WEB = Path("data"), Path("web")
try:  # in Colab: the key from Secrets, the data and page files uploaded
    from google.colab import files, userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
    for folder in (DATA, WEB):
        folder.mkdir(exist_ok=True)
    if not (DATA / "sample_cases.csv").exists():
        print("Upload sample_cases.csv and ai_assessments.json (data/), and index.html, app.js and "
              "styles.css (web/):")
        for name, content in files.upload().items():
            (WEB if name.endswith((".html", ".js", ".css")) else DATA).joinpath(name).write_bytes(content)
except ImportError:  # locally, from the project folder, with the key in .env
    from dotenv import load_dotenv
    load_dotenv(".env")
from groq import Groq

## 1. The cases, and what no single row shows
Besides its own row, the AI gets the other cases with the same claim number, how the amount,
visits and distance compare with the usual (median) for that type of care, and any hard evidence.

In [ ]:
cases = pd.read_csv(DATA / "sample_cases.csv")
records = {c["case_id"]: c for c in cases.to_dict("records")}
USUAL = cases.groupby("care_type")[["claim_amount_usd", "weekly_visit_frequency",
                                    "member_provider_distance_miles"]].median()
HARD = {"duplicate_service_billed": "the same service was billed twice",
        "service_overlap_other_provider": "another provider billed for the same time"}


def facts(case_id):
    c, usual = records[case_id], USUAL.loc[records[case_id]["care_type"]]
    twins = [o for o, r in records.items() if r["claim_number"] == c["claim_number"] and o != case_id]
    return {
        # A field named "peer" nudges the AI into jargon, so it is renamed.
        "case": {("referral_system_amount_vs_similar_pct" if k == "amount_vs_peer_avg_pct" else k): v
                 for k, v in c.items()},
        "shares_claim_number_with": twins,
        "typical_for_this_care_type": {f"typical_{k}": float(v) for k, v in usual.items()},
        # Pre-computed multiples: the AI misreads "+115%" and does arithmetic badly.
        "compared_with_usual": {f"{k}_times_usual": round(c[k] / v, 1) for k, v in usual.items()},
        "hard_evidence": [why for col, why in HARD.items() if c[col] == 1]
                         + [f"the same claim number is also used on {', '.join(twins)}"] * bool(twins),
    }

## 2. The AI: what it must return, its instructions, and the Groq call

In [ ]:
Level = Literal["Low", "Medium", "High"]
Action = Literal["Close as false positive", "Request documentation", "Escalate to SIU"]


class Fact(BaseModel):
    field: str  # a key copied from the facts, so it can be checked
    value: str
    why: str


class Assessment(BaseModel):
    summary: str
    key_facts: list[Fact]
    risk_score: int = Field(ge=0, le=100)
    risk_level: Level
    confidence: Level
    recommended_action: Action
    next_steps: list[str]
    open_questions: list[str]


class Answer(BaseModel):
    answer: str
    citations: list[Fact]
    answerable: bool


FACT = '{"field": str, "value": str, "why": str}'
SHAPES = {  # far fewer tokens than a JSON schema; pydantic still checks every reply
    Assessment: '{"summary": str, "key_facts": [' + FACT + '], "risk_score": int 0-100, '
                '"risk_level": "Low"|"Medium"|"High", "confidence": "Low"|"Medium"|"High", '
                '"recommended_action": "Close as false positive"|"Request documentation"|'
                '"Escalate to SIU", "next_steps": [str], "open_questions": [str]}',
    Answer: '{"answer": str, "citations": [' + FACT + '], "answerable": bool}',
}
WRITING = """
Write for a colleague who has never seen this data: plain, everyday words about what happened.
Say "the patient", "the provider", "similar cases", "usually". Data key names (with underscores)
never appear in your text, only in a cited "field", copied exactly from the facts, e.g.
{"field": "weekly_visit_frequency", "value": "12", "why": "..."}. Use dollars, miles and
percentages of bills (0.44 is "44% of the bills"). To compare with the usual, quote
compared_with_usual (2.2 means "about 2.2 times the usual"); never do your own arithmetic.
referral_system_amount_vs_similar_pct is the referral system's own comparison."""
ASSESS = """You are a junior fraud investigator at a long-term-care insurer, doing first-pass triage
of referrals from an automated fraud engine; most turn out to be false positives. A senior
investigator reviews your work. Use ONLY the facts given; never invent data, history or documents.
- risk_score is your 0-100 judgement of how likely this is fraud; risk_level must match it: under
  20 Low, 20-49 Medium, 50 or more High. confidence is how clear-cut the evidence is.
- Warning signs at 0 are normal; none at all is evidence for a false positive.
- If hard_evidence is not empty, never rate the case Low or recommend "Close as false positive".
- summary: at most 3 short sentences (about 60 words). At most 3 next_steps (concrete actions)
  and 3 open_questions (what this data can't answer), one short sentence each.""" + WRITING
QA = """You are a junior fraud investigator answering a senior investigator's questions about ONE
referral. Use ONLY the facts given: the case, its cross-case facts and your earlier assessment. If
they can't answer, say so, set answerable to false and name the record that would. Answer in 2-5
sentences and cite each fact you rely on. You advise; the investigator decides.""" + WRITING
MODELS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b"]  # the second takes over if the first is out


def ask_ai(system, user, schema):
    """A reply checked against `schema`. Tries each model with each key in GROQ_API_KEY
    (comma-separated, for more free quota); the SDK itself waits out per-minute rate limits."""
    error = None
    for model in MODELS:
        for key in os.environ["GROQ_API_KEY"].split(","):
            try:
                reply = Groq(api_key=key.strip(), max_retries=3).chat.completions.create(
                    model=model, temperature=0, reasoning_effort="low",
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": f"{system}\nReply with only JSON "
                                                            f"shaped like: {SHAPES[schema]}"},
                              {"role": "user", "content": user}])
                return schema.model_validate_json(reply.choices[0].message.content)
            except Exception as e:  # out of quota, bad key or invalid reply: try the next
                error = e
    raise RuntimeError(f"The AI is unavailable: {error}")

## 3. Assess every case, then check the AI's work
Results are saved to `data/ai_assessments.json`, so later runs (and anyone without a key) reuse
them; delete it to assess again. Every fact the AI cites is checked against the data, and a case
with hard evidence that the AI would close or rate Low is flagged.

In [ ]:
SAVED = DATA / "ai_assessments.json"
assessments = json.loads(SAVED.read_text()) if SAVED.exists() else {}


def assess(case_id):
    try:
        return case_id, ask_ai(ASSESS, f"Assess this referral.\n{json.dumps(facts(case_id))}",
                               Assessment).model_dump()
    except RuntimeError as e:
        print(case_id, e)
        return case_id, None


with ThreadPoolExecutor(4) as pool:  # a few at once; run the cell again to retry any that failed
    todo = [i for i in records if i not in assessments]
    assessments |= {i: a for i, a in pool.map(assess, todo) if a}
SAVED.write_text(json.dumps(assessments, indent=1))
print(f"{len(assessments)} of {len(records)} cases assessed")


def matches(cited, actual):
    """Does the value the AI cited match the data? Tolerates "$32,373", "205 miles", "44%" for 0.44."""
    cited = {"yes": "1", "true": "1", "no": "0", "false": "0"}.get(cited.strip().lower(), cited)
    if isinstance(actual, list):  # case ids; an empty list is a checkable "none"
        return all(str(x) in cited for x in actual) if actual else cited.strip().lower() in ("[]", "none", "0", "")
    if isinstance(actual, str):
        return cited.strip().lower() == actual.strip().lower()
    number = re.search(r"-?\d+(?:\.\d+)?", cited.replace(",", ""))
    return bool(number) and any(abs(x - actual) < 1e-6 for x in (float(number[0]), float(number[0]) / 100))


def check(case_id, cited):
    """Each cited fact, marked ok if the data confirms it ("a.b" paths count as "b")."""
    f = facts(case_id)
    data = {**records[case_id], **f["case"], **f["typical_for_this_care_type"],
            **f["compared_with_usual"], "shares_claim_number_with": f["shares_claim_number_with"],
            "hard_evidence": f["hard_evidence"]}
    return [{**c, "ok": (k := c["field"].split(".")[-1]) in data and matches(c["value"], data[k])}
            for c in cited]


def warnings(case_id):
    a = assessments[case_id]
    found = [f"The AI said {label(c['field']).lower()} is {c['value']}, which the data doesn't confirm"
             for c in check(case_id, a["key_facts"]) if not c["ok"]]
    if a["risk_level"] != ("High" if a["risk_score"] >= 50 else "Medium" if a["risk_score"] >= 20 else "Low"):
        found.append(f"The AI's score ({a['risk_score']}) doesn't match its risk level ({a['risk_level']})")
    if facts(case_id)["hard_evidence"] and (a["risk_level"] == "Low"
                                            or a["recommended_action"] == "Close as false positive"):
        found.append("The AI would close this case or rate it Low despite hard evidence")
    return found

## 4. Plain-English labels

In [ ]:
LABELS = {
    "case_id": "Case", "claim_number": "Claim number", "claim_date": "Claim date",
    "care_type": "Type of care", "claim_amount_usd": "Amount billed", "state": "State",
    "duplicate_service_billed": "Same service billed twice", "weekly_visit_frequency": "Visits per week",
    "member_provider_distance_miles": "Distance from patient to provider (miles)",
    "prior_claims_last_12mo": "Claims in the past year",
    "shared_contact_with_provider": "Patient and provider share contact details",
    "weekend_billing_ratio": "Bills on weekends", "round_dollar_billing_ratio": "Bills in round dollars",
    "amount_vs_peer_avg_pct": "Amount vs similar claims (%)",
    "referral_system_amount_vs_similar_pct": "Amount vs similar claims (%)",
    "recent_policy_change_flag": "Policy changed just before the claim",
    "service_overlap_other_provider": "Another provider billed for the same time",
    "shares_claim_number_with": "Same claim number also on", "hard_evidence": "Hard evidence",
}
FLAGS = {*HARD, "shared_contact_with_provider", "recent_policy_change_flag"}
SHORT = {"claim_amount_usd": "Amount", "weekly_visit_frequency": "Visits",
         "member_provider_distance_miles": "Distance"}


def label(field):
    field = field.split(".")[-1]
    base = field.removeprefix("typical_").removesuffix("_times_usual")
    if base != field:
        return f"{LABELS.get(base, base)}, {'typical' if field.startswith('typical_') else 'times the usual'}"
    return LABELS.get(field, field.replace("_", " ").capitalize())


def show(field, value):
    if isinstance(value, list):
        return ", ".join(map(str, value)) or "None"
    if field in FLAGS:
        return "Yes" if value else "No"
    if field.endswith("_ratio"):
        return f"{value:.0%} of bills"
    return f"${value:,.0f}" if field == "claim_amount_usd" else str(value)


def concerns(i):
    """What stands out, most serious first: hard evidence, yes/no red flags, big gaps from usual."""
    f, c, twins = facts(i), records[i], facts(i)["shares_claim_number_with"]
    return ([LABELS[k] for k in HARD if c[k] == 1] + [f"Same claim number as {', '.join(twins)}"] * bool(twins)
            + [LABELS[k] for k in ("shared_contact_with_provider", "recent_policy_change_flag") if c[k]]
            + [f"{SHORT[k[:-12]]} {v:g}× the usual" for k, v in f["compared_with_usual"].items() if v >= 2])

## 5. The app: a web page served from this notebook
Run this cell, then open the link. The queue is ordered by AI risk; open a case to see the AI's
assessment, ask it questions, accept or override its call (which moves the case to its next stage:
escalated, documents requested or closed) and leave notes. Decisions and notes are appended to
`outputs/audit.csv`, the audit trail (never edited). Re-run the cell to restart the app.

In [ ]:
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

PORT = 8000
AUDIT = Path("outputs/audit.csv")
COLUMNS = ["created_at", "case_id", "investigator", "decision", "final_risk_level", "final_action",
           "text", "ai_risk_level", "ai_action"]
STAGES = {"Escalate to SIU": "Escalated to SIU", "Request documentation": "Documents requested",
          "Close as false positive": "Closed"}
queue = sorted(assessments, key=lambda i: -assessments[i]["risk_score"])
WARN = {i: warnings(i) for i in queue}
lock = threading.Lock()


def trail(case_id=None):
    t = pd.read_csv(AUDIT, dtype=str).fillna("") if AUDIT.exists() else pd.DataFrame(columns=COLUMNS)
    rows = t.to_dict("records") if case_id is None else t[t.case_id == case_id].to_dict("records")
    for r in rows:
        r["stage"], r["reason"] = STAGES.get(r["final_action"], ""), r["text"]
    return rows


def latest():
    """The current decision per case: the most recent one (notes don't count)."""
    return {r["case_id"]: r for r in trail() if r["decision"] != "note"}


def log(case_id, investigator, decision, level="", action="", text=""):
    a = assessments[case_id]
    row = dict(zip(COLUMNS, [datetime.now().isoformat(timespec="seconds"), case_id, investigator, decision,
                             level, action, text, a["risk_level"], a["recommended_action"]]))
    with lock:
        AUDIT.parent.mkdir(exist_ok=True)
        pd.DataFrame([row]).to_csv(AUDIT, mode="a", header=not AUDIT.exists(), index=False)
    return row


def shown(case_id, cited):
    """Cited facts as the page shows them: a plain label, the value from the data, ✓ or ⚠."""
    data = {**records[case_id], **facts(case_id)["case"]}
    return [{**c, "verified": c["ok"], "label": label(c["field"]),
             "shown": show(k, data[k]) if c["ok"] and (k := c["field"].split(".")[-1]) in data else c["value"]}
            for c in check(case_id, cited)]


def clean_low(i):
    a = assessments[i]
    return (a["risk_level"] == "Low" and a["recommended_action"] == "Close as false positive"
            and not WARN[i] and not facts(i)["hard_evidence"])


def case_summary(i, decided):
    a, hard = assessments[i], facts(i)["hard_evidence"]
    return {"case_id": i, "tier": a["risk_level"], "score": a["risk_score"], "ai_status": "ok",
            "assessment": {**a, "key_indicators": shown(i, a["key_facts"])}, "grounding_issues": WARN[i],
            "floor_applied": bool(hard), "floor_reason": "; ".join(hard), "concerns": concerns(i),
            "shares_claim_number_with": facts(i)["shares_claim_number_with"],
            "clean_low": clean_low(i), "decision": decided.get(i)}


def case_detail(i):
    history = trail(i)
    c = records[i]
    return {**case_summary(i, latest()), "claim": c,
            "claim_display": [{"label": label(k), "value": show(k, v)} for k, v in c.items()],
            "history": {"decisions": [r for r in history if r["decision"] != "note"],
                        "notes": [r for r in history if r["decision"] == "note"]}}


def decide(i, body):
    a, who = assessments[i], (body.get("investigator") or "").strip()
    if not who:
        raise ValueError("Enter your name at the top first.")
    if body.get("decision") == "accept":
        return log(i, who, "accept", a["risk_level"], a["recommended_action"])
    level, action, reason = body.get("risk_level"), body.get("action"), (body.get("reason") or "").strip()
    if body.get("decision") != "override" or level not in Level.__args__ or action not in STAGES or not reason:
        raise ValueError("An override needs a risk level, an action and a reason")
    if (level, action) == (a["risk_level"], a["recommended_action"]):
        raise ValueError("That matches the AI's call; use Accept instead, or change the risk level or action")
    return log(i, who, "override", level, action, reason)


def add_note(i, body):
    who, text = (body.get("investigator") or "").strip(), (body.get("text") or "").strip()
    if not (who and text):
        raise ValueError("Enter your name and a note.")
    return log(i, who, "note", text=text)


def answer(i, body):
    question, a = (body.get("question") or "").strip(), assessments[i]
    if not question:
        raise ValueError("Ask a question.")
    earlier = [{"question": t["question"], "answer": t["answer"]} for t in body.get("history", [])[-6:]]
    gist = {k: a[k] for k in ("summary", "risk_score", "risk_level", "recommended_action")}
    reply = ask_ai(QA, f"Case facts:\n{json.dumps(facts(i))}\nYour earlier assessment:\n{json.dumps(gist)}"
                       f"\nEarlier questions:\n{json.dumps(earlier)}\nQuestion: {question}", Answer)
    return {**reply.model_dump(), "citations": shown(i, [c.model_dump() for c in reply.citations])}


def bulk_accept(body):
    who, decided = (body.get("investigator") or "").strip(), latest()
    if not who:
        raise ValueError("Enter your name at the top first.")
    accepted = [i for i in body.get("case_ids", []) if i in assessments and i not in decided and clean_low(i)]
    for i in accepted:
        log(i, who, "accept", "Low", "Close as false positive")
    return {"accepted": accepted, "skipped": [i for i in body.get("case_ids", []) if i not in accepted]}


class App(BaseHTTPRequestHandler):
    """The page (web/) and its JSON API."""
    TYPES = {".html": "text/html", ".js": "text/javascript", ".css": "text/css"}

    def send(self, body, status=200, kind="application/json"):
        data = body if isinstance(body, bytes) else json.dumps(body, default=str).encode()
        self.send_response(status)
        self.send_header("Content-Type", f"{kind}; charset=utf-8")
        self.send_header("Cache-Control", "no-cache")
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = self.path.split("?")[0]
        page = WEB / ("index.html" if path == "/" else Path(path).name)  # only files in web/
        if path == "/" or path.startswith("/static/") and page.is_file():
            return self.send(page.read_bytes(), kind=self.TYPES.get(page.suffix, "text/plain"))
        if path == "/api/cases":
            decided = latest()
            return self.send([case_summary(i, decided) for i in queue])
        if path.startswith("/api/cases/") and (i := path.split("/")[3]) in assessments:
            return self.send(case_detail(i))
        self.send({"detail": "Not found"}, 404)

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
        parts = self.path.split("?")[0].strip("/").split("/")  # api/cases/<id>/<action>
        try:
            if parts == ["api", "bulk-accept"]:
                return self.send(bulk_accept(body))
            action = {"decision": decide, "notes": add_note, "ask": answer}.get(parts[-1])
            if len(parts) != 4 or parts[:2] != ["api", "cases"] or parts[2] not in assessments or not action:
                return self.send({"detail": "Not found"}, 404)
            self.send(action(parts[2], body))
        except ValueError as e:
            self.send({"detail": str(e)}, 422)
        except RuntimeError as e:  # the AI is unavailable
            self.send({"detail": str(e)}, 502)

    def log_message(self, *args):  # keep the notebook's output quiet
        pass


if "server" in globals():  # re-running the cell restarts the app
    server.shutdown()
    server.server_close()
ThreadingHTTPServer.allow_reuse_address = True
server = ThreadingHTTPServer(("0.0.0.0", PORT), App)
threading.Thread(target=server.serve_forever, daemon=True).start()
try:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, anchor_text="Open the Case Review app")
except ImportError:
    print(f"The app is running: open http://localhost:{PORT}")